# UCSD Macro Optimizer
Constrained optimization of daily nutrition on a $28 dining dollar budget.

| Goal | Value |
|------|-------|
| Calories | ≥ 2,000 kcal |
| Protein | ≥ 150 g |
| Carbohydrates | ≥ 200 g |
| Fat | ≥ 60 g |
| Daily Budget | ≤ $28.00 |

Uses integer linear programming (PuLP/CBC) to minimize total daily spend while satisfying all macro constraints, with live scraping of UCSD dining hall menus and a curated database of campus market items.

---


In [ ]:
# Install dependencies if needed
import subprocess, sys
for pkg in ['pulp', 'requests', 'beautifulsoup4', 'plotly', 'lxml']:
    try:
        __import__(pkg.replace('-','_').replace('beautifulsoup4','bs4'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

# Core imports
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import pulp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
from IPython.display import display, HTML
import json, time, warnings, re
from datetime import datetime
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings('ignore')

try:
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    PLOTLY = True
except ImportError:
    PLOTLY = False

plt.rcParams.update({'figure.dpi': 120, 'font.family': 'sans-serif',
                     'axes.spines.top': False, 'axes.spines.right': False})
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.1f}'.format)

print(f'UCSD Macro Optimizer  |  {datetime.now().strftime("%Y-%m-%d")}')
print(f'PuLP {pulp.__version__}  |  Plotly: {PLOTLY}')

## 1. Configuration
Edit these values to change your goals, budget, or dietary restrictions.

In [ ]:
# Macro goals (minimum daily targets)
GOALS = {
    'calories': 2000,   # kcal
    'protein':   150,   # grams
    'carbs':     200,   # grams
    'fat':        60,   # grams
}

# Budget
BUDGET_CAP = 28.00  # hard ceiling in dining dollars

# Dietary filters (set True to exclude non-compliant items)
FILTERS = {
    'vegetarian': False,
    'vegan':      False,
    'no_pork':    False,
    'halal':      False,
    'gluten_free':False,
}

# LP solver settings
MAX_SERVINGS_DEFAULT = 3    # per item per day unless overridden
PENALTY_PER_GRAM     = 50   # penalty weight for unmet macro in soft-constraint fallback

print('Configuration loaded.')
print(f'  Goals  {GOALS}')
print(f'  Budget ${BUDGET_CAP:.2f}')
print(f'  Filters: {FILTERS}')

## 2. Food Database
### 2a. Campus Market Items
Curated from items available at Goody's Marketplace, Seventh Market, Sixth Market, and Sunshine Market. Prices reflect typical campus-store rates (2024-25).

In [ ]:
MARKET_ITEMS = [
    # Protein Shakes
    dict(name='Premier Protein Shake, Chocolate (11.5 oz)',
         category='Protein Shake', source='Market', location='All campus markets',
         price=4.50, serving_unit='1 bottle',
         calories=160, protein=30.0, carbs=5.0, fat=3.0, fiber=1.0, sodium=230,
         allergens=['milk','soy'], vegetarian=True, vegan=False, gluten_free=True,
         halal=False, contains_pork=False, max_daily=2,
         notes='User staple, 30g P for $4.50, best P/$ in shakes'),

    dict(name='Premier Protein Shake, Vanilla (11.5 oz)',
         category='Protein Shake', source='Market', location='All campus markets',
         price=4.50, serving_unit='1 bottle',
         calories=160, protein=30.0, carbs=5.0, fat=3.0, fiber=1.0, sodium=220,
         allergens=['milk','soy'], vegetarian=True, vegan=False, gluten_free=True,
         halal=False, contains_pork=False, max_daily=2,
         notes='Same macros as chocolate variant'),

    dict(name='Core Power Elite, Chocolate (11.5 oz)',
         category='Protein Shake', source='Market', location='All campus markets',
         price=4.99, serving_unit='1 bottle',
         calories=230, protein=42.0, carbs=8.0, fat=4.5, fiber=0.0, sodium=230,
         allergens=['milk'], vegetarian=True, vegan=False, gluten_free=True,
         halal=False, contains_pork=False, max_daily=2,
         notes='Highest protein density of any RTD shake'),

    dict(name='Muscle Milk Pro Series (14 oz)',
         category='Protein Shake', source='Market', location='Goody\'s, Seventh Market',
         price=4.79, serving_unit='1 bottle',
         calories=160, protein=25.0, carbs=10.0, fat=4.5, fiber=2.0, sodium=150,
         allergens=['milk','soy'], vegetarian=True, vegan=False, gluten_free=False,
         halal=False, contains_pork=False, max_daily=2,
         notes='Useful for hitting fat + protein together'),

    # Protein Bars
    dict(name='Quest Bar, Chocolate Chip Cookie Dough',
         category='Protein Bar', source='Market', location='All campus markets',
         price=3.79, serving_unit='1 bar (60g)',
         calories=190, protein=21.0, carbs=21.0, fat=8.0, fiber=14.0, sodium=150,
         allergens=['milk'], vegetarian=True, vegan=False, gluten_free=True,
         halal=False, contains_pork=False, max_daily=2,
         notes='Net carbs ~7g after fiber; solid P/$ ratio'),

    dict(name='RxBar, Chocolate Sea Salt',
         category='Protein Bar', source='Market', location='All campus markets',
         price=3.49, serving_unit='1 bar (52g)',
         calories=210, protein=12.0, carbs=23.0, fat=9.0, fiber=5.0, sodium=260,
         allergens=['eggs','tree nuts'], vegetarian=True, vegan=False, gluten_free=True,
         halal=True, contains_pork=False, max_daily=2,
         notes='Whole-food ingredients; halal-friendly'),

    dict(name='Kind Protein Bar, Dark Chocolate Nut',
         category='Protein Bar', source='Market', location='All campus markets',
         price=3.29, serving_unit='1 bar (50g)',
         calories=250, protein=12.0, carbs=24.0, fat=15.0, fiber=3.0, sodium=130,
         allergens=['tree nuts','peanuts'], vegetarian=True, vegan=True, gluten_free=True,
         halal=True, contains_pork=False, max_daily=2,
         notes='Good for fat macro; vegan + halal'),

    dict(name='Clif Bar, Chocolate Chip',
         category='Energy Bar', source='Market', location='All campus markets',
         price=2.49, serving_unit='1 bar (68g)',
         calories=240, protein=9.0, carbs=44.0, fat=5.0, fiber=5.0, sodium=160,
         allergens=['wheat','soy','tree nuts'], vegetarian=True, vegan=True, gluten_free=False,
         halal=False, contains_pork=False, max_daily=2,
         notes='Cheapest carb-dense bar; low protein'),

    # Dairy & Eggs
    dict(name='Chobani Greek Yogurt, Plain (5.3 oz)',
         category='Dairy', source='Market', location='All campus markets',
         price=2.49, serving_unit='1 cup (150g)',
         calories=90, protein=15.0, carbs=6.0, fat=0.0, fiber=0.0, sodium=55,
         allergens=['milk'], vegetarian=True, vegan=False, gluten_free=True,
         halal=True, contains_pork=False, max_daily=2,
         notes='Best P/$ in dairy; plain = lowest sugar'),

    dict(name='Chobani Greek Yogurt, Strawberry (5.3 oz)',
         category='Dairy', source='Market', location='All campus markets',
         price=2.49, serving_unit='1 cup (150g)',
         calories=130, protein=11.0, carbs=19.0, fat=0.0, fiber=0.0, sodium=70,
         allergens=['milk'], vegetarian=True, vegan=False, gluten_free=True,
         halal=True, contains_pork=False, max_daily=2,
         notes='Higher sugar/carbs than plain'),

    dict(name='Good Culture Cottage Cheese (5.3 oz)',
         category='Dairy', source='Market', location='Goody\'s, Seventh Market',
         price=3.49, serving_unit='1 cup (150g)',
         calories=120, protein=19.0, carbs=5.0, fat=2.5, fiber=0.0, sodium=380,
         allergens=['milk'], vegetarian=True, vegan=False, gluten_free=True,
         halal=False, contains_pork=False, max_daily=2,
         notes='Outstanding protein density for dairy'),

    dict(name='Hard Boiled Eggs, 2-Pack',
         category='Dairy', source='Market', location='Goody\'s, Seventh Market',
         price=2.99, serving_unit='2 eggs (100g)',
         calories=140, protein=12.0, carbs=1.0, fat=10.0, fiber=0.0, sodium=140,
         allergens=['eggs'], vegetarian=True, vegan=False, gluten_free=True,
         halal=True, contains_pork=False, max_daily=2,
         notes='Best fat source at markets; also protein'),

    dict(name='String Cheese, Sargento (1 oz)',
         category='Dairy', source='Market', location='All campus markets',
         price=1.49, serving_unit='1 stick (28g)',
         calories=80, protein=6.0, carbs=0.0, fat=6.0, fiber=0.0, sodium=200,
         allergens=['milk'], vegetarian=True, vegan=False, gluten_free=True,
         halal=False, contains_pork=False, max_daily=3,
         notes='Cheap protein + fat snack'),

    # Lean Protein
    dict(name='StarKist Tuna Packet (2.6 oz)',
         category='Protein', source='Market', location='All campus markets',
         price=2.49, serving_unit='1 packet (74g)',
         calories=70, protein=16.0, carbs=0.0, fat=0.5, fiber=0.0, sodium=210,
         allergens=['fish'], vegetarian=False, vegan=False, gluten_free=True,
         halal=True, contains_pork=False, max_daily=2,
         notes='#1 protein-per-dollar in the entire database'),

    # Nuts & Fats
    dict(name='Almonds, 1 oz Snack Pack',
         category='Nuts', source='Market', location='All campus markets',
         price=1.99, serving_unit='1 oz (28g)',
         calories=160, protein=6.0, carbs=6.0, fat=14.0, fiber=3.5, sodium=0,
         allergens=['tree nuts'], vegetarian=True, vegan=True, gluten_free=True,
         halal=True, contains_pork=False, max_daily=2,
         notes='Best fat source per dollar; vegan + halal'),

    dict(name='Justin\'s Almond Butter Packet',
         category='Nuts', source='Market', location='Goody\'s, Seventh Market',
         price=1.99, serving_unit='1 packet (32g)',
         calories=195, protein=7.0, carbs=8.0, fat=17.0, fiber=2.0, sodium=60,
         allergens=['tree nuts'], vegetarian=True, vegan=True, gluten_free=True,
         halal=True, contains_pork=False, max_daily=2,
         notes='Great fat macro booster; pairs with bagel or rice cakes'),

    # Prepared / Grab-and-Go
    dict(name='Goody\'s Breakfast Burrito',
         category='Prepared', source='Goody\'s Market', location='Goody\'s Marketplace',
         price=6.49, serving_unit='1 burrito (~300g)',
         calories=520, protein=25.0, carbs=48.0, fat=22.0, fiber=3.0, sodium=820,
         allergens=['eggs','wheat','milk'], vegetarian=False, vegan=False, gluten_free=False,
         halal=False, contains_pork=True, max_daily=1,
         notes='Current user staple, eggs, sausage, cheese, potato'),

    dict(name='Goody\'s Veggie Breakfast Burrito',
         category='Prepared', source='Goody\'s Market', location='Goody\'s Marketplace',
         price=6.49, serving_unit='1 burrito (~280g)',
         calories=470, protein=18.0, carbs=52.0, fat=17.0, fiber=5.0, sodium=720,
         allergens=['eggs','wheat','milk'], vegetarian=True, vegan=False, gluten_free=False,
         halal=True, contains_pork=False, max_daily=1,
         notes='No-meat variant; halal-friendly'),

    dict(name='Turkey & Swiss Sandwich',
         category='Prepared', source='Market', location='Goody\'s, Seventh Market',
         price=8.99, serving_unit='1 sandwich (~240g)',
         calories=440, protein=26.0, carbs=42.0, fat=14.0, fiber=2.0, sodium=950,
         allergens=['wheat','milk','eggs'], vegetarian=False, vegan=False, gluten_free=False,
         halal=False, contains_pork=False, max_daily=1,
         notes='Solid macros but expensive for the budget'),

    dict(name='Chicken Caesar Wrap',
         category='Prepared', source='Market', location='Goody\'s, Seventh Market',
         price=9.49, serving_unit='1 wrap (~300g)',
         calories=580, protein=32.0, carbs=48.0, fat=22.0, fiber=3.0, sodium=1050,
         allergens=['wheat','milk','eggs','fish'], vegetarian=False, vegan=False, gluten_free=False,
         halal=False, contains_pork=False, max_daily=1,
         notes='High protein but burns ~34% of daily budget'),

    dict(name='Tuna Salad Sandwich',
         category='Prepared', source='Market', location='Goody\'s, Seventh Market',
         price=7.99, serving_unit='1 sandwich (~240g)',
         calories=460, protein=22.0, carbs=44.0, fat=18.0, fiber=2.0, sodium=820,
         allergens=['wheat','fish','eggs'], vegetarian=False, vegan=False, gluten_free=False,
         halal=True, contains_pork=False, max_daily=1,
         notes='Cheaper than chicken wrap; halal-friendly'),

    # Fruit
    dict(name='Banana',
         category='Fruit', source='Market', location='All campus markets',
         price=0.99, serving_unit='1 medium (118g)',
         calories=105, protein=1.3, carbs=27.0, fat=0.4, fiber=3.1, sodium=1,
         allergens=[], vegetarian=True, vegan=True, gluten_free=True,
         halal=True, contains_pork=False, max_daily=3,
         notes='Cheapest carb on campus; ideal pre-workout'),

    dict(name='Apple',
         category='Fruit', source='Market', location='All campus markets',
         price=1.29, serving_unit='1 medium (182g)',
         calories=95, protein=0.5, carbs=25.0, fat=0.3, fiber=4.4, sodium=2,
         allergens=[], vegetarian=True, vegan=True, gluten_free=True,
         halal=True, contains_pork=False, max_daily=2,
         notes='Good fiber; slightly pricier than banana'),

    dict(name='Orange',
         category='Fruit', source='Market', location='All campus markets',
         price=1.29, serving_unit='1 medium (131g)',
         calories=62, protein=1.2, carbs=15.0, fat=0.2, fiber=3.1, sodium=0,
         allergens=[], vegetarian=True, vegan=True, gluten_free=True,
         halal=True, contains_pork=False, max_daily=2,
         notes='Vitamin C; lower cal than banana'),

    # Grains & Carbs
    dict(name='Oatmeal Cup, Instant (Quaker)',
         category='Grain', source='Market', location='All campus markets',
         price=2.29, serving_unit='1 cup (43g dry)',
         calories=150, protein=5.0, carbs=27.0, fat=3.0, fiber=3.0, sodium=100,
         allergens=['wheat'], vegetarian=True, vegan=True, gluten_free=False,
         halal=True, contains_pork=False, max_daily=2,
         notes='Great morning carb; just add water'),

    dict(name='Whole Grain Bagel',
         category='Grain', source='Market', location='Goody\'s, Seventh Market',
         price=2.49, serving_unit='1 bagel (98g)',
         calories=270, protein=10.0, carbs=52.0, fat=2.0, fiber=4.0, sodium=410,
         allergens=['wheat'], vegetarian=True, vegan=True, gluten_free=False,
         halal=True, contains_pork=False, max_daily=2,
         notes='Dense carb source; pairs with almond butter'),

    dict(name='Rice Cakes, Plain (3-pack)',
         category='Grain', source='Market', location='All campus markets',
         price=1.99, serving_unit='3 cakes (30g)',
         calories=105, protein=2.0, carbs=22.0, fat=0.5, fiber=0.5, sodium=60,
         allergens=[], vegetarian=True, vegan=True, gluten_free=True,
         halal=True, contains_pork=False, max_daily=2,
         notes='GF carb option; pair with tuna or nut butter'),

    # Beverages
    dict(name='Chocolate Milk (14 fl oz)',
         category='Beverage', source='Market', location='All campus markets',
         price=2.99, serving_unit='1 bottle (414ml)',
         calories=290, protein=8.0, carbs=48.0, fat=7.0, fiber=0.0, sodium=250,
         allergens=['milk'], vegetarian=True, vegan=False, gluten_free=True,
         halal=True, contains_pork=False, max_daily=2,
         notes='Classic post-workout recovery drink'),

    dict(name='Orange Juice (12 fl oz)',
         category='Beverage', source='Market', location='All campus markets',
         price=2.99, serving_unit='1 bottle (355ml)',
         calories=170, protein=2.0, carbs=40.0, fat=0.0, fiber=0.0, sodium=10,
         allergens=[], vegetarian=True, vegan=True, gluten_free=True,
         halal=True, contains_pork=False, max_daily=1,
         notes='Fast carb spike; high sugar'),

    dict(name='Drip Coffee, Large (16 oz)',
         category='Beverage', source='Dining Hall', location='All dining locations',
         price=2.00, serving_unit='1 cup (475ml)',
         calories=5, protein=0.3, carbs=1.0, fat=0.0, fiber=0.0, sodium=5,
         allergens=[], vegetarian=True, vegan=True, gluten_free=True,
         halal=True, contains_pork=False, max_daily=2,
         notes='Negligible macros but essential'),
]

print(f'Market database: {len(MARKET_ITEMS)} items loaded.')

### 2b. Dining Hall Items
A la carte prices estimated from typical UCSD HDH dining rates. These appear in the optimizer alongside market items.

In [ ]:
DINING_ITEMS = [
    # Proteins
    dict(name='Grilled Chicken Breast (4 oz)',
         category='Protein', source='Dining Hall', location='64 Degrees, OceanView',
         price=6.50, serving_unit='4 oz (113g)',
         calories=165, protein=31.0, carbs=0.0, fat=3.5, fiber=0.0, sodium=360,
         allergens=[], vegetarian=False, vegan=False, gluten_free=True,
         halal=False, contains_pork=False, max_daily=2,
         notes='Best protein per serving at dining halls'),

    dict(name='Salmon Fillet (4 oz)',
         category='Protein', source='Dining Hall', location='OceanView, 64 Degrees',
         price=9.50, serving_unit='4 oz (113g)',
         calories=180, protein=25.0, carbs=0.0, fat=8.5, fiber=0.0, sodium=380,
         allergens=['fish'], vegetarian=False, vegan=False, gluten_free=True,
         halal=True, contains_pork=False, max_daily=1,
         notes='Great omega-3s; helps hit fat goal'),

    dict(name='Scrambled Eggs (1 cup)',
         category='Protein', source='Dining Hall', location='64 Degrees, Canyon Vista (breakfast)',
         price=3.50, serving_unit='1 cup (220g)',
         calories=230, protein=16.0, carbs=2.0, fat=17.0, fiber=0.0, sodium=420,
         allergens=['eggs','milk'], vegetarian=True, vegan=False, gluten_free=True,
         halal=True, contains_pork=False, max_daily=2,
         notes='Cheap protein + fat at breakfast'),

    dict(name='Beef Burger Patty (4 oz, no bun)',
         category='Protein', source='Dining Hall', location='64 Degrees (Triton Grill), Foodworx',
         price=7.50, serving_unit='1 patty (113g)',
         calories=290, protein=23.0, carbs=0.0, fat=21.0, fiber=0.0, sodium=450,
         allergens=[], vegetarian=False, vegan=False, gluten_free=True,
         halal=False, contains_pork=False, max_daily=1,
         notes='High fat + protein; skip bun to stay low-carb'),

    dict(name='Tofu Stir Fry (1 serving)',
         category='Protein', source='Dining Hall', location='64 Degrees, OceanView',
         price=7.00, serving_unit='1 serving (~200g)',
         calories=250, protein=14.0, carbs=22.0, fat=12.0, fiber=3.0, sodium=580,
         allergens=['soy'], vegetarian=True, vegan=True, gluten_free=False,
         halal=True, contains_pork=False, max_daily=1,
         notes='Best vegan protein at dining halls'),

    dict(name='Breakfast Bowl (eggs + potatoes + cheese)',
         category='Prepared', source='Dining Hall', location='Canyon Vista, OceanView (breakfast)',
         price=7.00, serving_unit='1 bowl (~300g)',
         calories=460, protein=20.0, carbs=40.0, fat=22.0, fiber=3.0, sodium=760,
         allergens=['eggs','milk'], vegetarian=True, vegan=False, gluten_free=True,
         halal=True, contains_pork=False, max_daily=1,
         notes='Good breakfast when dining halls open early'),

    dict(name='Cheese Burger (with bun)',
         category='Prepared', source='Dining Hall', location='64 Degrees (Triton Grill), Foodworx',
         price=9.50, serving_unit='1 burger (~280g)',
         calories=620, protein=32.0, carbs=48.0, fat=30.0, fiber=2.0, sodium=1100,
         allergens=['wheat','milk','eggs'], vegetarian=False, vegan=False, gluten_free=False,
         halal=False, contains_pork=False, max_daily=1,
         notes='Hits fat + protein but uses 34% of budget'),

    dict(name='Cheese Pizza (2 slices)',
         category='Prepared', source='Dining Hall', location='Foodworx, OceanView',
         price=6.00, serving_unit='2 slices (~200g)',
         calories=500, protein=18.0, carbs=68.0, fat=14.0, fiber=2.0, sodium=1000,
         allergens=['wheat','milk'], vegetarian=True, vegan=False, gluten_free=False,
         halal=False, contains_pork=False, max_daily=1,
         notes='Good carb+protein balance but high sodium'),

    # Starches
    dict(name='White Rice (1 cup cooked)',
         category='Starch', source='Dining Hall', location='All dining halls',
         price=2.50, serving_unit='1 cup (186g)',
         calories=200, protein=4.0, carbs=44.0, fat=0.5, fiber=0.5, sodium=10,
         allergens=[], vegetarian=True, vegan=True, gluten_free=True,
         halal=True, contains_pork=False, max_daily=3,
         notes='Cheapest carb at dining halls'),

    dict(name='Brown Rice (1 cup cooked)',
         category='Starch', source='Dining Hall', location='All dining halls',
         price=2.50, serving_unit='1 cup (195g)',
         calories=220, protein=5.0, carbs=46.0, fat=2.0, fiber=3.5, sodium=10,
         allergens=[], vegetarian=True, vegan=True, gluten_free=True,
         halal=True, contains_pork=False, max_daily=3,
         notes='More fiber; interchangeable with white rice'),

    dict(name='Pasta with Marinara (1 cup)',
         category='Starch', source='Dining Hall', location='64 Degrees (al Dente)',
         price=5.50, serving_unit='1 cup (250g)',
         calories=320, protein=11.0, carbs=62.0, fat=3.0, fiber=4.0, sodium=480,
         allergens=['wheat'], vegetarian=True, vegan=True, gluten_free=False,
         halal=True, contains_pork=False, max_daily=2,
         notes='Good carb source at 64 Degrees al Dente station'),

    # Legumes
    dict(name='Black Beans (1/2 cup)',
         category='Legume', source='Dining Hall', location='64 Degrees (Taqueria), OceanView',
         price=2.50, serving_unit='1/2 cup (86g)',
         calories=110, protein=7.0, carbs=20.0, fat=0.5, fiber=7.5, sodium=200,
         allergens=[], vegetarian=True, vegan=True, gluten_free=True,
         halal=True, contains_pork=False, max_daily=2,
         notes='Plant protein + fiber; great filler for taqueria bowls'),

    dict(name='Pinto Beans (1/2 cup)',
         category='Legume', source='Dining Hall', location='64 Degrees (Taqueria)',
         price=2.50, serving_unit='1/2 cup (86g)',
         calories=115, protein=7.0, carbs=22.0, fat=0.5, fiber=6.0, sodium=200,
         allergens=[], vegetarian=True, vegan=True, gluten_free=True,
         halal=True, contains_pork=False, max_daily=2,
         notes='Interchangeable with black beans at taqueria'),

    # Vegetables
    dict(name='Steamed Broccoli (1 cup)',
         category='Vegetable', source='Dining Hall', location='All dining halls',
         price=2.00, serving_unit='1 cup (156g)',
         calories=55, protein=4.0, carbs=11.0, fat=0.5, fiber=5.0, sodium=65,
         allergens=[], vegetarian=True, vegan=True, gluten_free=True,
         halal=True, contains_pork=False, max_daily=3,
         notes='Volume + fiber without many calories'),

    dict(name='Garden Salad (large)',
         category='Vegetable', source='Dining Hall', location='64 Degrees (Garden Bar)',
         price=5.50, serving_unit='1 large (~300g)',
         calories=80, protein=3.0, carbs=14.0, fat=1.0, fiber=5.0, sodium=100,
         allergens=[], vegetarian=True, vegan=True, gluten_free=True,
         halal=True, contains_pork=False, max_daily=1,
         notes='Low-cal base; add protein toppings'),
]

print(f'Dining hall database: {len(DINING_ITEMS)} items loaded.')

### 2c. Live Scraper for the UCSD Dining Portal
Attempts to pull today's menu from active dining halls. Falls back gracefully if the site is unavailable or no halls are open.

In [ ]:
BASE_URL = 'https://hdh-web.ucsd.edu/dining/apps/diningservices'

VENUES = [
    {'name': '64 Degrees',          'locId': 64,  'locDetID': 18},
    {'name': 'Bistro',              'locId': 27,  'locDetID': 13},
    {'name': 'Canyon Vista',        'locId': 24,  'locDetID': 11},
    {'name': 'Club Med',            'locId': 15,  'locDetID':  7},
    {'name': 'Foodworx',            'locId': 11,  'locDetID':  6},
    {'name': 'OceanView',           'locId':  5,  'locDetID':  4},
    {'name': 'Sixth College',       'locId': 37,  'locDetID': 24},
    {'name': 'Ventanas',            'locId': 18,  'locDetID':  8},
]

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
    'Referer': BASE_URL,
}

def fetch_nutrition(item_id: str, rec_id: str, session: requests.Session) -> Optional[dict]:
    """Fetch one item's nutrition from the UCSD nutrition facts page."""
    url = f'{BASE_URL}/Nutrition/Nutritionfacts2?id={item_id}&recId={rec_id}'
    try:
        r = session.get(url, headers=HEADERS, timeout=10)
        if r.status_code != 200:
            return None
        soup = BeautifulSoup(r.text, 'lxml')
        # Item name
        h1 = soup.find('h1')
        name = h1.get_text(strip=True) if h1 else f'Item {rec_id}'
        # Nutrition table rows
        rows = soup.find_all('tr')
        nutrients = {}
        for row in rows:
            cells = row.find_all(['td','th'])
            if len(cells) >= 2:
                label = cells[0].get_text(strip=True).lower()
                value = re.sub(r'[^\d.]', '', cells[1].get_text(strip=True))
                if value:
                    nutrients[label] = float(value)
        def g(key, *alts):
            for k in [key] + list(alts):
                if k in nutrients:
                    return nutrients[k]
            return 0.0
        return dict(
            name=name,
            category='Dining Hall (Scraped)',
            source='Dining Hall',
            location='UCSD Dining',
            price=None,  # prices not in nutrition page; see note below
            serving_unit='1 serving',
            calories=g('calories', 'total calories'),
            protein=g('protein'),
            carbs=g('total carbohydrates', 'carbohydrates', 'total carbs'),
            fat=g('total fat', 'fat'),
            fiber=g('dietary fiber', 'fiber'),
            sodium=g('sodium'),
            allergens=[],
            vegetarian=False,
            vegan=False,
            gluten_free=False,
            halal=False,
            contains_pork=False,
            max_daily=2,
            notes=f'Live-scraped from UCSD dining (id={item_id}, recId={rec_id})'
        )
    except Exception as e:
        return None


def scrape_venue_menu(venue: dict, day_num: int, session: requests.Session) -> List[dict]:
    """Try to get menu items from a venue page.
    UCSD venue pages are JS-rendered; this attempts static HTML parsing
    and extracts nutrition links when present.
    """
    url = (f'{BASE_URL}/Restaurants/Venue_V3'
           f'?locId={venue["locId"]}&locDetID={venue["locDetID"]}&dayNum={day_num}')
    try:
        r = session.get(url, headers=HEADERS, timeout=12)
        if r.status_code != 200:
            return []
        soup = BeautifulSoup(r.text, 'lxml')
        # Search for nutrition-fact links containing id= and recId=
        links = soup.find_all('a', href=re.compile(r'Nutritionfacts2.*id=.*recId=', re.I))
        items = []
        seen = set()
        for link in links:
            href = link.get('href', '')
            m_id  = re.search(r'id=([^&]+)', href)
            m_rec = re.search(r'recId=([^&]+)', href)
            if m_id and m_rec:
                key = (m_id.group(1), m_rec.group(1))
                if key not in seen:
                    seen.add(key)
                    item = fetch_nutrition(m_id.group(1), m_rec.group(1), session)
                    if item:
                        item['location'] = venue['name']
                        items.append(item)
                        time.sleep(0.3)  # be polite to the server
        return items
    except Exception:
        return []


def scrape_all_venues(day_num: int = 0, verbose: bool = True) -> List[dict]:
    """Scrape all venues for a given day (0=Sun … 6=Sat). Returns list of items."""
    session = requests.Session()
    all_items = []
    for venue in VENUES:
        items = scrape_venue_menu(venue, day_num, session)
        if verbose:
            status = f'{len(items)} items' if items else 'closed / not scraped'
            print(f'  {venue["name"]:25s}  →  {status}')
        all_items.extend(items)
    return all_items


print('Scraper defined. Run scrape_all_venues() to pull live dining hall menus.')
print('(UCSD pages are JS-rendered; static scraping may return 0 items when halls are closed)')

In [ ]:
# Attempt live scrape (today = Sunday = dayNum 0)
today_day_num = datetime.now().weekday()  # Mon=0 … Sun=6 in Python; UCSD uses 0=Sun
ucsd_day_num  = (today_day_num + 1) % 7   # shift to UCSD convention

print(f'Scraping today\'s menus (dayNum={ucsd_day_num})...')
SCRAPED_ITEMS = scrape_all_venues(day_num=ucsd_day_num, verbose=True)

# Drop scraped items without prices (can't include in optimizer)
SCRAPED_PRICED = [i for i in SCRAPED_ITEMS if i.get('price') is not None]
print(f'\nScraped {len(SCRAPED_ITEMS)} items total; {len(SCRAPED_PRICED)} have prices.')
print('Proceeding with curated database + any priced scraped items.')

## 3. Build Combined Food DataFrame

In [ ]:
# Combine all sources
all_items = MARKET_ITEMS + DINING_ITEMS + SCRAPED_PRICED

df = pd.DataFrame(all_items)
df = df.dropna(subset=['price', 'calories', 'protein', 'carbs', 'fat'])
df = df[df['price'] > 0].reset_index(drop=True)

# Derived columns
df['protein_per_dollar'] = df['protein']  / df['price']
df['calories_per_dollar']= df['calories'] / df['price']
df['fat_per_dollar']     = df['fat']      / df['price']
df['carbs_per_dollar']   = df['carbs']    / df['price']
df['cal_per_gram_protein'] = df.apply(
    lambda r: r['calories'] / r['protein'] if r['protein'] > 0 else np.inf, axis=1)

print(f'Combined database: {len(df)} items')
print(f'  Sources  {df["source"].value_counts().to_dict()}')
print(f'  Categories: {df["category"].nunique()} unique')
print()

display(df[['name','source','price','calories','protein','carbs','fat',
            'protein_per_dollar']].sort_values('protein_per_dollar', ascending=False)
         .head(10).style.format({'price': '${:.2f}', 'calories': '{:.0f}',
                                  'protein': '{:.1f}g', 'carbs': '{:.1f}g',
                                  'fat': '{:.1f}g', 'protein_per_dollar': '{:.1f}g/$'})
         .set_caption('Top 10 items by protein per dollar'))

## 4. Exploratory Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('UCSD Campus Food Macro Efficiency Overview', fontsize=15, fontweight='bold', y=1.01)

# Panel 1: Protein per dollar by category
ax = axes[0, 0]
cat_p = df.groupby('category')['protein_per_dollar'].mean().sort_values(ascending=True)
bars = ax.barh(cat_p.index, cat_p.values, color=sns.color_palette('husl', len(cat_p)))
ax.set_xlabel('Avg Protein per Dollar (g/$)')
ax.set_title('Protein Efficiency by Category')
for bar, val in zip(bars, cat_p.values):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=8)

# Panel 2: Calories vs Protein scatter
ax = axes[0, 1]
cats = df['category'].unique()
palette = dict(zip(cats, sns.color_palette('husl', len(cats))))
for cat, grp in df.groupby('category'):
    ax.scatter(grp['protein'], grp['calories'],
               label=cat, c=[palette[cat]], alpha=0.75, s=60, edgecolors='white')
ax.axvline(x=150/4, color='red', linestyle='--', alpha=0.4, label='150g P split over ~4 items')
ax.set_xlabel('Protein (g)')
ax.set_ylabel('Calories (kcal)')
ax.set_title('Protein vs Calories per Serving')
ax.legend(fontsize=7, ncol=2, loc='upper left')

# Panel 3: Price distribution by source
ax = axes[1, 0]
for src, grp in df.groupby('source'):
    ax.hist(grp['price'], bins=10, alpha=0.6, label=src)
ax.axvline(x=BUDGET_CAP, color='red', linestyle='--', label=f'${BUDGET_CAP} budget')
ax.set_xlabel('Price per Serving ($)')
ax.set_ylabel('Count')
ax.set_title('Price Distribution by Source')
ax.legend()

# Panel 4, top 15 items by protein per dollar
ax = axes[1, 1]
top15 = df.nlargest(15, 'protein_per_dollar')[['name','protein_per_dollar','source']]
colors = ['#2196F3' if s == 'Market' else '#FF9800' for s in top15['source']]
ax.barh(top15['name'].str[:40], top15['protein_per_dollar'], color=colors)
ax.set_xlabel('Protein per Dollar (g/$)')
ax.set_title('Top 15 Items: Protein per Dollar')
legend_patches = [mpatches.Patch(color='#2196F3', label='Market'),
                  mpatches.Patch(color='#FF9800', label='Dining Hall')]
ax.legend(handles=legend_patches, fontsize=8)

plt.tight_layout()
plt.savefig('analysis_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: analysis_overview.png')

## 5. Optimization Engine

**Formulation:**
```
Minimize:   Σ (price_i · x_i)                 [total daily cost]

Subject to:
  Σ (calories_i · x_i)  ≥  2000              [calorie floor]
  Σ (protein_i  · x_i)  ≥  150               [protein floor]
  Σ (carbs_i    · x_i)  ≥  200               [carb floor]
  Σ (fat_i      · x_i)  ≥  60                [fat floor]
  Σ (price_i    · x_i)  ≤  28                [budget cap]
  x_i ∈ {0, 1, … max_daily_i}                [integer servings]
```

If the hard-constraint problem is **infeasible**, a soft-constraint fallback is run that penalizes unmet goals so the closest-possible plan is always returned.

In [ ]:
def apply_dietary_filters(food_df: pd.DataFrame, filters: dict) -> pd.DataFrame:
    """Return subset of food_df that passes all active dietary filters."""
    mask = pd.Series(True, index=food_df.index)
    if filters.get('vegetarian'):
        mask &= food_df['vegetarian'].astype(bool)
    if filters.get('vegan'):
        mask &= food_df['vegan'].astype(bool)
    if filters.get('no_pork'):
        mask &= ~food_df['contains_pork'].astype(bool)
    if filters.get('halal'):
        mask &= food_df['halal'].astype(bool)
    if filters.get('gluten_free'):
        mask &= food_df['gluten_free'].astype(bool)
    return food_df[mask].copy()


def run_optimizer(
    food_df: pd.DataFrame,
    goals: dict,
    budget_cap: float,
    filters: dict = None,
    verbose: bool = True,
) -> Tuple[pd.DataFrame, dict]:
    """Run ILP optimizer.  Returns (plan_df, summary_dict)."""

    # Apply dietary filters
    if filters:
        food_df = apply_dietary_filters(food_df, filters)

    if food_df.empty:
        raise ValueError('No food items remain after applying dietary filters.')

    idx = list(food_df.index)

    # Decision variables
    prob = pulp.LpProblem('UCSD_Macro_Optimizer', pulp.LpMinimize)
    x = {i: pulp.LpVariable(f'x_{i}', lowBound=0,
                              upBound=int(food_df.loc[i, 'max_daily']),
                              cat='Integer')
         for i in idx}

    # Objective: minimize cost
    prob += pulp.lpSum(food_df.loc[i, 'price'] * x[i] for i in idx), 'Total_Cost'

    # Hard constraints
    prob += (pulp.lpSum(food_df.loc[i, 'calories'] * x[i] for i in idx)
             >= goals['calories'], 'min_calories')
    prob += (pulp.lpSum(food_df.loc[i, 'protein']  * x[i] for i in idx)
             >= goals['protein'],  'min_protein')
    prob += (pulp.lpSum(food_df.loc[i, 'carbs']    * x[i] for i in idx)
             >= goals['carbs'],    'min_carbs')
    prob += (pulp.lpSum(food_df.loc[i, 'fat']      * x[i] for i in idx)
             >= goals['fat'],      'min_fat')
    prob += (pulp.lpSum(food_df.loc[i, 'price']    * x[i] for i in idx)
             <= budget_cap,        'budget_cap')

    # Solve
    solver = pulp.PULP_CBC_CMD(msg=0, timeLimit=30)
    status = prob.solve(solver)
    feasible = pulp.LpStatus[prob.status] == 'Optimal'

    if not feasible:
        if verbose:
            print(f'Hard-constraint problem: {pulp.LpStatus[prob.status]}.')
            print('Running soft-constraint fallback (goals become targets, not hard limits)...')
        return _soft_fallback(food_df, goals, budget_cap, verbose=verbose)

    # Extract plan
    plan_rows = []
    for i in idx:
        n = int(round(x[i].varValue or 0))
        if n > 0:
            row = food_df.loc[i].copy()
            row['servings']   = n
            row['total_cost'] = row['price'] * n
            row['total_cal']  = row['calories'] * n
            row['total_prot'] = row['protein']  * n
            row['total_carb'] = row['carbs']    * n
            row['total_fat']  = row['fat']      * n
            plan_rows.append(row)

    plan_df = pd.DataFrame(plan_rows)
    summary = _build_summary(plan_df, goals, budget_cap, feasible=True)
    return plan_df, summary


def _soft_fallback(food_df, goals, budget_cap, verbose):
    """Soft-constraint fallback: minimize cost + penalty * shortfall."""
    idx = list(food_df.index)
    prob = pulp.LpProblem('UCSD_Soft_Optimizer', pulp.LpMinimize)
    x = {i: pulp.LpVariable(f'x_{i}', lowBound=0,
                              upBound=int(food_df.loc[i, 'max_daily']),
                              cat='Integer') for i in idx}
    s_cal  = pulp.LpVariable('slack_cal',  lowBound=0)
    s_prot = pulp.LpVariable('slack_prot', lowBound=0)
    s_carb = pulp.LpVariable('slack_carb', lowBound=0)
    s_fat  = pulp.LpVariable('slack_fat',  lowBound=0)

    P = PENALTY_PER_GRAM
    prob += (pulp.lpSum(food_df.loc[i, 'price'] * x[i] for i in idx)
             + P * s_cal + P * s_prot + P * s_carb + P * s_fat)

    prob += (pulp.lpSum(food_df.loc[i, 'calories'] * x[i] for i in idx) + s_cal
             >= goals['calories'])
    prob += (pulp.lpSum(food_df.loc[i, 'protein']  * x[i] for i in idx) + s_prot
             >= goals['protein'])
    prob += (pulp.lpSum(food_df.loc[i, 'carbs']    * x[i] for i in idx) + s_carb
             >= goals['carbs'])
    prob += (pulp.lpSum(food_df.loc[i, 'fat']      * x[i] for i in idx) + s_fat
             >= goals['fat'])
    prob += (pulp.lpSum(food_df.loc[i, 'price']    * x[i] for i in idx)
             <= budget_cap)

    prob.solve(pulp.PULP_CBC_CMD(msg=0, timeLimit=30))

    plan_rows = []
    for i in idx:
        n = int(round(x[i].varValue or 0))
        if n > 0:
            row = food_df.loc[i].copy()
            row['servings']   = n
            row['total_cost'] = row['price'] * n
            row['total_cal']  = row['calories'] * n
            row['total_prot'] = row['protein']  * n
            row['total_carb'] = row['carbs']    * n
            row['total_fat']  = row['fat']      * n
            plan_rows.append(row)

    plan_df = pd.DataFrame(plan_rows)
    summary = _build_summary(plan_df, goals, budget_cap, feasible=False)
    return plan_df, summary


def _build_summary(plan_df, goals, budget_cap, feasible):
    """Compute achieved macros and budget, compare to goals."""
    achieved = {
        'cost':     plan_df['total_cost'].sum() if not plan_df.empty else 0,
        'calories': plan_df['total_cal'].sum()  if not plan_df.empty else 0,
        'protein':  plan_df['total_prot'].sum() if not plan_df.empty else 0,
        'carbs':    plan_df['total_carb'].sum() if not plan_df.empty else 0,
        'fat':      plan_df['total_fat'].sum()  if not plan_df.empty else 0,
    }
    gaps = {
        k: max(0, goals.get(k, 0) - achieved.get(k, 0))
        for k in ['calories', 'protein', 'carbs', 'fat']
    }
    return dict(achieved=achieved, gaps=gaps, goals=goals,
                budget_cap=budget_cap, feasible=feasible)


print('Optimizer functions defined.')

## 6. Run the Optimization

In [ ]:
print('=' * 60)
print('Solving UCSD Macro Optimizer')
print('=' * 60)
print(f'  Goals  {GOALS}')
print(f'  Budget ${BUDGET_CAP:.2f}')
print(f'  Items  {len(df)} available')
print()

plan_df, summary = run_optimizer(df, GOALS, BUDGET_CAP, filters=FILTERS, verbose=True)

a = summary['achieved']
g = summary['goals']
gaps = summary['gaps']

status_label = 'OPTIMAL' if summary['feasible'] else 'BEST-EFFORT (goals partially unmet)'
print(f'\nStatus {status_label}')
print()
print(f"  Total cost   ${a['cost']:.2f} / ${BUDGET_CAP:.2f} budget")
print(f"  Calories     {a['calories']:.0f} / {g['calories']} kcal  "
      + (f'[UNDER by {gaps["calories"]:.0f}]' if gaps['calories'] else '[OK]'))
print(f"  Protein      {a['protein']:.1f}g / {g['protein']}g  "
      + (f'[UNDER by {gaps["protein"]:.1f}g]' if gaps['protein'] else '[OK]'))
print(f"  Carbs        {a['carbs']:.1f}g / {g['carbs']}g  "
      + (f'[UNDER by {gaps["carbs"]:.1f}g]' if gaps['carbs'] else '[OK]'))
print(f"  Fat          {a['fat']:.1f}g / {g['fat']}g  "
      + (f'[UNDER by {gaps["fat"]:.1f}g]' if gaps['fat'] else '[OK]'))


## 7. Your Optimal Daily Meal Plan

In [ ]:
if plan_df.empty:
    print('No plan found. Check budget and database.')
else:
    display_cols = ['name', 'servings', 'serving_unit', 'price',
                    'total_cost', 'total_cal', 'total_prot', 'total_carb', 'total_fat']
    display_df = plan_df[display_cols].copy()
    display_df.columns = ['Food Item', 'Qty', 'Unit', 'Unit Price',
                           'Cost', 'Calories', 'Protein (g)', 'Carbs (g)', 'Fat (g)']

    # Totals row
    totals = pd.DataFrame([['** DAILY TOTAL **', '', '', '',
                             plan_df['total_cost'].sum(),
                             plan_df['total_cal'].sum(),
                             plan_df['total_prot'].sum(),
                             plan_df['total_carb'].sum(),
                             plan_df['total_fat'].sum()]],
                           columns=display_df.columns)

    display_df = pd.concat([display_df, totals], ignore_index=True)

    style = (
        display_df.style
        .format({'Unit Price': '${:.2f}', 'Cost': '${:.2f}',
                 'Calories': '{:.0f}', 'Protein (g)': '{:.1f}',
                 'Carbs (g)': '{:.1f}', 'Fat (g)': '{:.1f}'}, na_rep='')
        .apply(lambda col: ['background-color: #e8f5e9; font-weight: bold'
                            if col.name in ['Food Item'] and col == '** DAILY TOTAL **'
                            else '' for _ in col], axis=0)
        .set_properties(**{'text-align': 'left'})
        .set_caption('Optimal Daily Meal Plan')
        .hide(axis='index')
    )
    display(style)

## 8. Results Visualization

In [ ]:
if not plan_df.empty:
    fig = plt.figure(figsize=(16, 10))
    gs  = GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.4)
    fig.suptitle('Optimal Daily Meal Plan for the UCSD Macro Optimizer',
                 fontsize=14, fontweight='bold')

    # Panel 1: Cost breakdown pie
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.pie(plan_df['total_cost'],
            labels=[n[:25] + '…' if len(n) > 25 else n for n in plan_df['name']],
            autopct='%1.0f%%', startangle=140,
            colors=sns.color_palette('husl', len(plan_df)),
            textprops={'fontsize': 8})
    ax1.set_title(f'Cost Breakdown\n(Total: ${plan_df["total_cost"].sum():.2f})')

    # Panel 2: Macro breakdown pie
    ax2 = fig.add_subplot(gs[0, 1])
    macro_cal = {
        'Protein': plan_df['total_prot'].sum() * 4,
        'Carbs':   plan_df['total_carb'].sum() * 4,
        'Fat':     plan_df['total_fat'].sum()  * 9,
    }
    ax2.pie(macro_cal.values(), labels=macro_cal.keys(),
            autopct='%1.1f%%', startangle=90,
            colors=['#2196F3','#FF9800','#4CAF50'])
    ax2.set_title(f'Calories by Macro\n(Total: {plan_df["total_cal"].sum():.0f} kcal)')

    # Panel 3: Goals vs achieved bar chart
    ax3 = fig.add_subplot(gs[0, 2])
    macros = ['calories', 'protein', 'carbs', 'fat']
    labels = ['Calories', 'Protein (g)', 'Carbs (g)', 'Fat (g)']
    achieved_vals = [summary['achieved'][m] for m in macros]
    goal_vals     = [summary['goals'][m] for m in macros]
    pcts = [100 * a / g if g else 0 for a, g in zip(achieved_vals, goal_vals)]
    colors_ = ['#4CAF50' if p >= 100 else '#F44336' for p in pcts]
    bars = ax3.barh(labels, pcts, color=colors_, alpha=0.8)
    ax3.axvline(x=100, color='black', linestyle='--', lw=1)
    for bar, pct, av, gv in zip(bars, pcts, achieved_vals, goal_vals):
        ax3.text(min(pct, 100) + 1, bar.get_y() + bar.get_height()/2,
                 f'{av:.0f} / {gv:.0f}', va='center', fontsize=8)
    ax3.set_xlabel('% of Goal Achieved')
    ax3.set_title('Goals vs Achieved')
    ax3.set_xlim(0, max(max(pcts) + 10, 110))

    # Panel 4: Protein per item stacked bar
    ax4 = fig.add_subplot(gs[1, :])
    short_names = [n[:30] + '…' if len(n) > 30 else n for n in plan_df['name']]
    x_pos = np.arange(len(plan_df))
    width = 0.25
    b1 = ax4.bar(x_pos - width, plan_df['total_prot'], width, label='Protein', color='#2196F3')
    b2 = ax4.bar(x_pos,          plan_df['total_carb'], width, label='Carbs',   color='#FF9800')
    b3 = ax4.bar(x_pos + width, plan_df['total_fat'],  width, label='Fat',     color='#4CAF50')
    ax4.set_xticks(x_pos)
    ax4.set_xticklabels(short_names, rotation=30, ha='right', fontsize=9)
    ax4.set_ylabel('Grams')
    ax4.set_title('Macros Contributed per Food Item')
    ax4.legend()
    # Goal lines
    n_items = len(plan_df)
    ax4.axhline(y=GOALS['protein']/n_items,  color='#2196F3', linestyle=':', alpha=0.5,
                label=f'Protein goal / item')

    plt.savefig('optimal_plan.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: optimal_plan.png')

## 9. vs. Your Current Diet

In [ ]:
# What you've been eating (user's stated current diet)
CURRENT_DIET = [
    {'name': 'Premier Protein Shake (Chocolate)',
     'qty': 2, 'price': 4.50,
     'calories': 160, 'protein': 30.0, 'carbs': 5.0, 'fat': 3.0},
    {'name': "Goody's Breakfast Burrito",
     'qty': 1, 'price': 6.49,
     'calories': 520, 'protein': 25.0, 'carbs': 48.0, 'fat': 22.0},
]

def summarize_diet(diet_list):
    totals = {'cost': 0, 'calories': 0, 'protein': 0, 'carbs': 0, 'fat': 0}
    for item in diet_list:
        q = item['qty']
        totals['cost']     += item['price']    * q
        totals['calories'] += item['calories'] * q
        totals['protein']  += item['protein']  * q
        totals['carbs']    += item['carbs']    * q
        totals['fat']      += item['fat']      * q
    return totals

current_totals  = summarize_diet(CURRENT_DIET)
optimal_totals  = summary['achieved']

comparison = pd.DataFrame({
    'Metric': ['Cost ($)', 'Calories (kcal)', 'Protein (g)', 'Carbs (g)', 'Fat (g)'],
    'Current Diet': [
        current_totals['cost'], current_totals['calories'],
        current_totals['protein'], current_totals['carbs'], current_totals['fat']
    ],
    'Optimal Plan': [
        optimal_totals['cost'], optimal_totals['calories'],
        optimal_totals['protein'], optimal_totals['carbs'], optimal_totals['fat']
    ],
    'Goal': [
        BUDGET_CAP, GOALS['calories'], GOALS['protein'], GOALS['carbs'], GOALS['fat']
    ],
})
comparison['Delta'] = comparison['Optimal Plan'] - comparison['Current Diet']
comparison['% of Goal (Current)']  = (comparison['Current Diet'] / comparison['Goal'] * 100).round(1)
comparison['% of Goal (Optimal)'] = (comparison['Optimal Plan'] / comparison['Goal'] * 100).round(1)

display(comparison.style
        .format({'Current Diet': '{:.1f}', 'Optimal Plan': '{:.1f}',
                 'Goal': '{:.1f}', 'Delta': '{:+.1f}',
                 '% of Goal (Current)': '{:.0f}%', '% of Goal (Optimal)': '{:.0f}%'})
        .set_caption('Current Diet vs Optimal Plan vs Goals')
        .hide(axis='index'))

print()
print('Key insight')
cal_gap = GOALS['calories'] - current_totals['calories']
prot_gap = GOALS['protein'] - current_totals['protein']
print(f"  Your current diet gives only {current_totals['calories']:.0f} kcal/day, "
      f"under by {cal_gap:.0f} kcal ({cal_gap/GOALS['calories']*100:.0f}% of your goal).")
print(f"  Protein {current_totals['protein']:.0f}g of {GOALS['protein']}g target "
      f"({prot_gap:.0f}g short).")
if current_totals['cost'] > optimal_totals['cost']:
    print(f"  Daily spend ${current_totals['cost']:.2f} vs optimal ${optimal_totals['cost']:.2f}, "
          f"a savings of ${current_totals['cost'] - optimal_totals['cost']:.2f}/day.")
else:
    print(f"  Daily spend ${current_totals['cost']:.2f} vs optimal ${optimal_totals['cost']:.2f}.")

In [ ]:
# Comparison chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Current Diet vs Optimal Plan vs Goals', fontsize=13, fontweight='bold')

metrics   = ['Calories', 'Protein (g)', 'Carbs (g)', 'Fat (g)']
goal_vals = [GOALS['calories'], GOALS['protein'], GOALS['carbs'], GOALS['fat']]
curr_vals = [current_totals['calories'], current_totals['protein'],
             current_totals['carbs'], current_totals['fat']]
opt_vals  = [optimal_totals['calories'], optimal_totals['protein'],
             optimal_totals['carbs'], optimal_totals['fat']]

x = np.arange(len(metrics))
w = 0.25

ax = axes[0]
ax.bar(x - w, [c/g*100 for c, g in zip(curr_vals, goal_vals)], w, label='Current', color='#F44336', alpha=0.8)
ax.bar(x,     [o/g*100 for o, g in zip(opt_vals, goal_vals)],  w, label='Optimal', color='#4CAF50', alpha=0.8)
ax.axhline(y=100, color='black', linestyle='--', lw=1.2, label='100% of goal')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel('% of Daily Goal Achieved')
ax.set_title('Goal Achievement (%)')
ax.legend()

ax = axes[1]
cost_data = [current_totals['cost'], optimal_totals['cost'], BUDGET_CAP]
labels_   = ['Current Diet', 'Optimal Plan', 'Budget Cap']
bar_colors= ['#F44336', '#4CAF50', '#9E9E9E']
bars = ax.bar(labels_, cost_data, color=bar_colors, alpha=0.8)
for bar, val in zip(bars, cost_data):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.3,
            f'${val:.2f}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Daily Spend ($)')
ax.set_title('Daily Cost Comparison')
ax.set_ylim(0, BUDGET_CAP + 5)

plt.tight_layout()
plt.savefig('comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Dietary Restriction Analysis
Run the optimizer under each restriction to see how your options change.

In [ ]:
SCENARIOS = [
    {'label': 'No Restrictions',  'filters': {}},
    {'label': 'Vegetarian',       'filters': {'vegetarian': True}},
    {'label': 'No Pork',          'filters': {'no_pork': True}},
    {'label': 'Halal',            'filters': {'halal': True}},
    {'label': 'Gluten-Free',      'filters': {'gluten_free': True}},
    {'label': 'Vegan',            'filters': {'vegan': True}},
]

results = []
for sc in SCENARIOS:
    print(f"  Running: {sc['label']}...")
    try:
        _, summ = run_optimizer(df.copy(), GOALS, BUDGET_CAP, filters=sc['filters'], verbose=False)
        a = summ['achieved']
        results.append({
            'Diet': sc['label'],
            'Cost ($)': a['cost'],
            'Calories': a['calories'],
            'Protein (g)': a['protein'],
            'Carbs (g)': a['carbs'],
            'Fat (g)': a['fat'],
            'All Goals Met': summ['feasible'],
            'Items Available': len(apply_dietary_filters(df, sc['filters'])),
        })
    except Exception as e:
        results.append({'Diet': sc['label'], 'Error': str(e)})

diet_df = pd.DataFrame(results)
display(diet_df.style
        .format({'Cost ($)': '${:.2f}', 'Calories': '{:.0f}',
                 'Protein (g)': '{:.1f}', 'Carbs (g)': '{:.1f}', 'Fat (g)': '{:.1f}'})
        .applymap(lambda v: 'color: green; font-weight: bold' if v is True else
                             'color: red' if v is False else '',
                  subset=['All Goals Met'])
        .set_caption('Optimizer Results Under Different Dietary Restrictions')
        .hide(axis='index'))

## 11. Budget Sweep Sensitivity Analysis

In [ ]:
budgets = np.arange(10, 30, 2)  # $10 to $28 in $2 steps
sens_results = []

print('Running budget sweep...')
for b in budgets:
    _, summ = run_optimizer(df.copy(), GOALS, float(b), filters={}, verbose=False)
    a = summ['achieved']
    sens_results.append({
        'Budget ($)': b,
        'Cost ($)':   a['cost'],
        'Calories':   a['calories'],
        'Protein (g)':a['protein'],
        'Carbs (g)':  a['carbs'],
        'Fat (g)':    a['fat'],
        'Feasible':   summ['feasible'],
    })
    print(f'  ${b:.0f}: cost=${a["cost"]:.2f}, '
          f'P={a["protein"]:.0f}g, cal={a["calories"]:.0f}, '
          f'feasible={summ["feasible"]}')

sens_df = pd.DataFrame(sens_results)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('How Macros Improve as Budget Increases',
             fontsize=13, fontweight='bold')

for ax, (col, goal_val, label, color) in zip(
    axes.flatten(),
    [('Calories',    GOALS['calories'], 'kcal', '#FF9800'),
     ('Protein (g)', GOALS['protein'],  'g',    '#2196F3'),
     ('Carbs (g)',   GOALS['carbs'],    'g',    '#4CAF50'),
     ('Fat (g)',     GOALS['fat'],      'g',    '#9C27B0')]
):
    ax.plot(sens_df['Budget ($)'], sens_df[col], 'o-', color=color, lw=2)
    ax.axhline(y=goal_val, color='red', linestyle='--', lw=1, label=f'Goal: {goal_val}{label}')
    ax.axvline(x=BUDGET_CAP, color='gray', linestyle=':', lw=1, label=f'Your budget: ${BUDGET_CAP}')
    ax.fill_between(sens_df['Budget ($)'], sens_df[col], goal_val,
                    where=(sens_df[col] < goal_val), alpha=0.15, color='red', label='Shortfall')
    ax.set_xlabel('Budget ($)')
    ax.set_ylabel(f'{col} ({label})')
    ax.set_title(col)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

# Find minimum budget to hit all goals
feasible_rows = sens_df[sens_df['Feasible'] == True]
if not feasible_rows.empty:
    min_budget = feasible_rows['Budget ($)'].min()
    print(f'\nMinimum budget to hit ALL goals: ~${min_budget:.0f}/day')
else:
    print('\nNo budget in the $10-$28 range achieves all goals simultaneously.')
    print('Consider adjusting your goals (see Section 12).')

## 12. Achievable Goal Targets for a $28 Budget

In [ ]:
# Sweep protein goal while keeping other goals fixed
protein_targets = list(range(80, 175, 10))
protein_sweep = []

print('Sweeping protein goal (other goals fixed)...')
for p_goal in protein_targets:
    g_ = {**GOALS, 'protein': p_goal}
    _, summ = run_optimizer(df.copy(), g_, BUDGET_CAP, filters={}, verbose=False)
    a = summ['achieved']
    protein_sweep.append({
        'Protein Goal (g)': p_goal,
        'Achieved Protein (g)': a['protein'],
        'Achieved Calories': a['calories'],
        'Cost ($)': a['cost'],
        'Feasible': summ['feasible'],
    })

psweep_df = pd.DataFrame(protein_sweep)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Protein Goal Sweep ($28 budget fixed)', fontsize=13, fontweight='bold')

colors = ['#4CAF50' if f else '#F44336' for f in psweep_df['Feasible']]
ax1.bar(psweep_df['Protein Goal (g)'].astype(str), psweep_df['Achieved Protein (g)'],
        color=colors, alpha=0.8)
ax1.plot(psweep_df['Protein Goal (g)'].astype(str), psweep_df['Protein Goal (g)'],
         'k--', lw=1, label='Target line')
ax1.set_xlabel('Protein Goal (g)')
ax1.set_ylabel('Achieved Protein (g)')
ax1.set_title('Green = goal met within $28')
ax1.legend(fontsize=8)
green_patch = mpatches.Patch(color='#4CAF50', label='All goals met')
red_patch   = mpatches.Patch(color='#F44336', label='Goals not met')
ax1.legend(handles=[green_patch, red_patch], fontsize=8)

ax2.plot(psweep_df['Protein Goal (g)'], psweep_df['Cost ($)'], 'o-', color='#2196F3')
ax2.axhline(y=BUDGET_CAP, color='red', linestyle='--', label=f'${BUDGET_CAP} cap')
ax2.set_xlabel('Protein Goal (g)')
ax2.set_ylabel('Optimal Cost ($)')
ax2.set_title('Cost vs Protein Target')
ax2.legend()

plt.tight_layout()
plt.savefig('goal_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

feasible_targets = psweep_df[psweep_df['Feasible'] == True]['Protein Goal (g)']
if not feasible_targets.empty:
    print(f'Maximum achievable protein within $28: ~{feasible_targets.max()}g/day')
else:
    print('Even 80g protein cannot be achieved alongside other goals for $28.')

## 13. Add or Update Food Items
Add market items you've found or update prices when they change.

In [ ]:
def add_item(name, price, calories, protein, carbs, fat,
             category='Other', source='Market', location='',
             fiber=0, sodium=0, allergens=None, serving_unit='1 serving',
             vegetarian=False, vegan=False, gluten_free=False,
             halal=False, contains_pork=False, max_daily=2, notes=''):
    """Add a new food item to the database. Call this then re-run the optimizer cells."""
    global df
    new = dict(
        name=name, category=category, source=source, location=location,
        price=price, serving_unit=serving_unit,
        calories=calories, protein=protein, carbs=carbs, fat=fat,
        fiber=fiber, sodium=sodium, allergens=allergens or [],
        vegetarian=vegetarian, vegan=vegan, gluten_free=gluten_free,
        halal=halal, contains_pork=contains_pork, max_daily=max_daily, notes=notes,
        protein_per_dollar=protein/price if price else 0,
        calories_per_dollar=calories/price if price else 0,
        fat_per_dollar=fat/price if price else 0,
        carbs_per_dollar=carbs/price if price else 0,
        cal_per_gram_protein=calories/protein if protein else float('inf'),
    )
    df = pd.concat([df, pd.DataFrame([new])], ignore_index=True)
    print(f'Added {name}, ${price:.2f}, {calories} kcal, {protein}g P, {carbs}g C, {fat}g F')
    return new


# Example: add an item you found on campus
# add_item(
#     name='Kirkland Greek Yogurt (from CVS near campus)',
#     price=2.00, calories=110, protein=17, carbs=8, fat=0,
#     category='Dairy', source='Market', location='CVS near UCSD',
#     vegetarian=True, gluten_free=True, halal=True, max_daily=2,
#     notes='Cheaper than Chobani; found at the CVS on Gilman'
# )

print('add_item() is ready. Uncomment the example above and edit values to add your own items.')
print(f'Current database: {len(df)} items')

## 14. Summary and Recommendations

What the optimizer shows

1. The current diet (2 Premier Protein shakes plus a Goody's burrito) hits only about 840 kcal and 85g protein, roughly 42% of the calorie goal and 57% of the protein goal. That's a large shortfall.

2. The best protein per dollar items at UCSD are the Premier Protein Shake (6.7 g/\$), the StarKist Tuna Packet (6.4 g/\$), and plain Chobani Greek Yogurt (6.0 g/\$).

3. Fat is the hardest macro to hit cheaply on campus. The cheapest fat sources are almonds ($1.99/pack, 14g fat) and hard boiled eggs ($2.99 for 10g fat).

4. The sensitivity analysis shows the minimum budget needed to hit all four goals simultaneously. If it's above $28, the notebook shows which goals to relax first.

Quick wins without changing your whole routine

- Replace one Premier Protein shake with a Tuna Packet plus Chobani combo. Same cost (about $5), similar protein (31g vs 30g), adds variety.
- Add 3 bananas to the current daily diet. $3, +315 cal, +81g carbs, closes most of the calorie gap at almost zero cost.
- Add 1 almond packet ($1.99) to start hitting the fat goal.

---
Prices are estimates based on 2024-25 campus market rates. Update them in the database using `add_item()` or by editing the MARKET_ITEMS / DINING_ITEMS lists above.
